In [ ]:
import os
import torch
import numpy as np
os.environ["KERAS_BACKEND"] = "torch"

In [ ]:
if torch.backends.mps.is_available():
    print("Apple's MPS backend (used by PyTorch on M1/M2 Macs) does not support float64 (double precision).")
    mps_enabled = True
    torch.set_default_dtype(torch.float32)
    print("set default to float32")

In [ ]:
import keras
import tensorflow as tf

In [ ]:
if keras.backend.backend() != "torch":
    print(f"warning: keras backend is set to {keras.backend.backend()}, restart jupyter kernel!!!!")
    raise RuntimeError()

In [ ]:
keras.backend.backend()

In [ ]:
import json

global config
with open('./config/keras_nn.json') as keras_nn_config:
    config = json.load(keras_nn_config)
    print("config loaded")

In [ ]:
# Root-level fields
batch_size = config["batchSize"]
scaler_enabled = config["scaler"]["enabled"]
scaler_type = config["scaler"]["type"]
input_size = config["inputSize"]
output_size = config["outputSize"]
seed = config["seed"]

In [ ]:
keras.utils.set_random_seed(seed)

In [ ]:
%load_ext tensorboard
# now available at http://localhost:6006/?

In [ ]:
# Dataset initialization

from utils.data_loader import get_ml_cup_data, split_dataloader, cv_fold_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler

def _scaler():
    if not scaler_enabled:
        return None

    match scaler_type:
        case "Standard":
            return StandardScaler()
        case "MinMax":
            return MinMaxScaler()
        case "Robust":
            return RobustScaler()
        case "MaxAbsScaler":
            return MaxAbsScaler()
        case _:
            return None

train_loader, test_loader = get_ml_cup_data(
    batch_size, 
    scaler=_scaler(),
    mps=mps_enabled
    )

In [ ]:
train_loader.dataset.X.shape, train_loader.dataset.y.shape

In [ ]:
test_loader.dataset.X.shape, test_loader.dataset.y.shape

In [ ]:
test_loader.dataset.y.shape[1]

In [ ]:
import numpy as np

y_mean = train_loader.dataset.y.mean(axis=0)        # (4,)

y_pred_baseline = np.tile(y_mean, (len(train_loader.dataset.y), 1))

mee_errors = np.linalg.norm(train_loader.dataset.y - y_pred_baseline, axis=1)
mse_errors = np.square(train_loader.dataset.y - y_pred_baseline)
mae_errors = np.abs(train_loader.dataset.y - y_pred_baseline)

mee_baseline = mee_errors.mean()
mse_baseline = mse_errors.mean()

print("Baseline MEE:", mee_baseline)
print("Baseline MSE:", mse_baseline)

In [ ]:
from losses import MeanEuclidianError

mee = MeanEuclidianError(name="mee", dtype=torch.float32)

In [ ]:
train_dataset = train_loader.dataset
test_dataset = test_loader.dataset

In [ ]:
import utils.keras as ukeras

## Randomized Search

In [ ]:
from scipy.stats import loguniform

param_distributions = {
    "reg__model__learning_rate": loguniform(1e-3, 1e-2),
    "reg__model__lambda_1": loguniform(3e-3, 1e-1),
    "reg__model__lambda_2": loguniform(3e-3, 1e-1),
    "reg__model__activation_1": ["relu", "gelu", "leaky_relu"],
    "reg__model__activation_2": ["relu", "gelu", "leaky_relu"],
    "reg__model__dropout_1": loguniform(0.2, 0.5),
    "reg__model__dropout_2": loguniform(0.2, 0.5),
}

In [ ]:
from executors import RandomizedSearchRegressionExecutor

# using default param_distribution
rs_executor = RandomizedSearchRegressionExecutor(
    train_loader=train_loader,
    units=[(16,8), 4],
    use_PCA=True,
    pca_input_size=2,
    n_iter=1,
    epochs=1,
    loss="mean_squared_error",
    baseline=mse_baseline,
    scoring="neg_mean_squared_error",
    seed=seed,
    save_path="keras/models/rs/regression"
)

In [ ]:
rs_executor.execute()

## Optuna

In [ ]:
# def general_objective(trial, u, path, train_loader_cv, validation_loader_cv):
#     unit1, unit2 = u

#     # Suggest hyperparameters
#     lambda_1 = trial.suggest_float('lambda_1', 3e-3, 1e-1, log=True)
#     learning_rate = trial.suggest_float('learning_rate', 1e-3, 1e-2, log=True)
#     activation_1 = trial.suggest_categorical('activation_1', ["relu", "gelu", "leaky_relu"])
#     dropout_1 = trial.suggest_float('dropout_1', 0.2, 0.5, log=True)

#     # Build model
#     if unit2 is not None:
#         lambda_2 = trial.suggest_float('lambda_2', 3e-3, 1e-1, log=True)
#         activation_2 = trial.suggest_categorical('activation_2', ["relu", "gelu", "leaky_relu"])
#         dropout_2 = trial.suggest_float('dropout_2', 0.2, 0.5, log=True)
#         model = build_model_two_hidden(
#             unit1=unit1,
#             unit2=unit2,
#             learning_rate=learning_rate,
#             lambda_1=lambda_1,
#             lambda_2=lambda_2,
#             activation_1=activation_1,
#             activation_2=activation_2,
#             dropout_1=dropout_1,
#             dropout_2=dropout_2
#         )
#     else:
#         model = build_model_single_hidden(
#             unit1=unit1,
#             learning_rate=learning_rate,
#             lambda_1=lambda_1,
#             activation_1=activation_1,
#             dropout_1=dropout_1,
#         )

#     pruning_cb = optuna.integration.KerasPruningCallback(trial, "val_loss")
#     checkpoint_path = f"{path}/checkpoints/optuna_trial_{trial.number}.keras"
#     checkpoint_cb = keras.callbacks.ModelCheckpoint(
#         filepath=checkpoint_path,
#         monitor="val_loss",
#         mode="min",
#         save_best_only=True,
#         save_weights_only=False
#     )
    
#     # Train model
#     history = model.fit(
#         train_loader_cv,
#         validation_data=validation_loader_cv,
#         epochs=1500,
#         verbose=0,
#         callbacks=[pruning_cb, checkpoint_cb]
#     )

#     val_loss = history.history["val_loss"][-1]
#     return val_loss

In [ ]:
# reduced_dataset = apply_pca_on_X(train_dataset, pca_input_size, standardize=False)

In [ ]:
# for u in units:
#     print(f"===unit {u}===")
#     unit1, unit2 = u

#     name = str(unit1)
#     if unit2 is not None:
#         name += "x" + str(unit2)

#     optuna_unitspecific_path = f"{optuna_base_path}/{name}"
#     ## optuna
#     def objective_cv(trial):
#         fold = KFold(n_splits=5, shuffle=True, random_state=seed)
#         scores = []
#         for fold_idx, (train_idx, valid_idx) in enumerate(fold.split(range(len(reduced_dataset)))):
#             print(f"Executing fold {fold_idx}")
#             train_loader_cv, validation_loader_cv = cv_fold_split(reduced_dataset, train_idx, valid_idx, 80)
#             accuracy = general_objective(trial, u, optuna_unitspecific_path, train_loader_cv, validation_loader_cv)
#             scores.append(accuracy)
#         return np.mean(scores)
    
    
#     study = optuna.create_study(study_name="keras-ML-CUP-"+name, sampler=sampler, direction="minimize")
#     study.optimize(objective_cv, n_trials=100)
    
#     df = study.trials_dataframe()
#     optuna_results_path = f"{optuna_base_path}/{name}/optuna_results.csv"
#     df.to_csv(optuna_results_path, index=False)

#     with open(optuna_base_path+"/hp.json", "w+") as f:
#         json.dump(study.best_trial.params, f, indent=2)